# Genetic Algorithm — Q1: User-Defined Function

This notebook keeps the original Q1 Genetic Algorithm workflow:
Initial Population → Fitness → Selection Probability → Expected Count → Actual Count →
Mating Pool → Crossover → Mutation → Population 2 → Convergence Check.

Only the objective function is entered by the user.
The original Q1 mating/crossover point remains fixed: **cut after bit 3**.


### A.1 Setup — encode / decode / user-defined fitness function


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sympy as sp

CHROM_LENGTH = 5   # 5 bits -> x in [0, 31]

# USER INPUT: function only
function_expression_q1 = input("Enter Q1 function f(x) (example: x^2): ")

x_symbol = sp.symbols('x')
expr_q1 = sp.sympify(function_expression_q1.replace("^", "**"))
fitness_q1 = sp.lambdify(x_symbol, expr_q1, "math")

def decode(chromosome: str) -> int:
    return int(chromosome, 2)

def crossover(parent1: str, parent2: str, point: int):
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2

def mutate_at(chromosome: str, position: int) -> str:
    bits = list(chromosome)
    bits[position] = '1' if bits[position] == '0' else '0'
    return ''.join(bits)

def build_generation_table(population, fitness_fn, mode='max'):
    d = pd.DataFrame({'Chromosome': population})
    d['x'] = d['Chromosome'].apply(decode)
    d['f(x)'] = d['x'].apply(fitness_fn)

    if mode == 'max':
        d['Probability'] = (d['f(x)'] / d['f(x)'].sum()).round(3)
        d['Expected count'] = (d['f(x)'] / d['f(x)'].mean()).round(3)
    else:
        worst = d['f(x)'].max()
        d['Selection fitness'] = worst - d['f(x)'] + 1e-9
        d['Probability'] = (d['Selection fitness'] / d['Selection fitness'].sum()).round(3)
        d['Expected count'] = (d['Probability'] * len(d)).round(3)

    d['Actual count'] = d['Expected count'].round().astype(int)
    return d

def mating_pool_from(table):
    pool = []
    for _, row in table.iterrows():
        pool.extend([row['Chromosome']] * int(row['Actual count']))
    return pool

print("Q1 function:", function_expression_q1)


Q1 function: x^2


### A.2 Population 1 (Initial) — fitness, probability, expected/actual count

In [2]:

population_1_q1 = ['11011', '10001', '01111', '10111']

table1_q1 = build_generation_table(population_1_q1, fitness_q1, mode='max')
print(f"Sum f(x) = {table1_q1['f(x)'].sum()}   Average f(x) = {table1_q1['f(x)'].mean():.1f}   Max f(x) = {table1_q1['f(x)'].max()}")
table1_q1


Sum f(x) = 1772   Average f(x) = 443.0   Max f(x) = 729


,Chromosome,x,f(x),Probability,Expected count,Actual count
0,11011,27,729,0.411,1.646,2
1,10001,17,289,0.163,0.652,1
2,01111,15,225,0.127,0.508,1
3,10111,23,529,0.299,1.194,1


### A.3 Mating pool (from Population 1)

In [3]:

pool_q1 = mating_pool_from(table1_q1)
print("Mating pool:", pool_q1)


Mating pool: ['11011', '11011', '10001', '01111', '10111']


### A.4 Crossover → A.5 Mutation → **Population 2**

The original Q1 mating/crossover point is retained: **cut after bit 3**.


In [4]:
pairs_q1 = [(pool_q1[0], pool_q1[1]), (pool_q1[2], pool_q1[3])]
cut = 3   # ORIGINAL Q1 MATING/CROSSOVER POINT

offspring_q1 = []
for p1, p2 in pairs_q1:
    c1, c2 = crossover(p1, p2, cut)
    offspring_q1.extend([c1, c2])

print("After crossover:", offspring_q1)

# ORIGINAL Q1 mutation positions retained
offspring_q1[0] = mutate_at(offspring_q1[0], 2)
offspring_q1[1] = mutate_at(offspring_q1[1], 3)

population_2_q1 = offspring_q1
print("Population 2:", population_2_q1)


After crossover: ['11011', '11011', '10011', '01101']
Population 2: ['11111', '11001', '10011', '01101']


### A.6 Population 2 — fitness, probability, expected/actual count

In [5]:

table2_q1 = build_generation_table(population_2_q1, fitness_q1, mode='max')
print(f"Sum f(x) = {table2_q1['f(x)'].sum()}   Average f(x) = {table2_q1['f(x)'].mean():.1f}   Max f(x) = {table2_q1['f(x)'].max()}")
table2_q1


Sum f(x) = 2116   Average f(x) = 529.0   Max f(x) = 961


,Chromosome,x,f(x),Probability,Expected count,Actual count
0,11111,31,961,0.454,1.817,2
1,11001,25,625,0.295,1.181,1
2,10011,19,361,0.171,0.682,1
3,01101,13,169,0.080,0.319,0


### A.7 Convergence check — Population 1 vs Population 2 (Q1)

In [6]:

def convergence_report(table1, table2, mode, label, true_optimum=None):
    best1 = table1['f(x)'].max() if mode == 'max' else table1['f(x)'].min()
    best2 = table2['f(x)'].max() if mode == 'max' else table2['f(x)'].min()
    div1, div2 = table1['Chromosome'].nunique(), table2['Chromosome'].nunique()
    var1, var2 = table1['f(x)'].var(ddof=0), table2['f(x)'].var(ddof=0)

    improved = (best2 > best1) if mode == 'max' else (best2 < best1)

    print(f"--- Convergence Report: {label} ---")
    print(f"{'Metric':<22}{'Population 1':>15}{'Population 2':>15}")
    print(f"{'Best f(x)':<22}{best1:>15.4f}{best2:>15.4f}")
    print(f"{'Diversity (unique)':<22}{div1:>15}{div2:>15}")
    print(f"{'Fitness variance':<22}{var1:>15.4f}{var2:>15.4f}")
    if true_optimum is not None:
        print(f"{'True optimum f(x)':<22}{'':>15}{true_optimum:>15.4f}")

    print()
    if not improved and (div2 <= 2 or var2 < 1e-3):
        verdict = "CONVERGED (or converging): best value stopped improving and diversity/variance has collapsed."
    elif improved and div2 == len(table2):
        verdict = "NOT YET CONVERGED: best value is still improving and full diversity remains -- continue running more generations."
    else:
        verdict = "PARTIALLY CONVERGED: some signals point to convergence, others don't -- run a few more generations to be sure."
    print("Verdict:", verdict)
    return verdict

verdict_q1 = convergence_report(table1_q1, table2_q1, mode='max', label='Q1 (maximize x^2)', true_optimum=961)


--- Convergence Report: Q1 (maximize x^2) ---
Metric                   Population 1   Population 2
Best f(x)                    729.0000       961.0000
Diversity (unique)                  4              4
Fitness variance           40108.0000     88416.0000
True optimum f(x)                           961.0000

Verdict: NOT YET CONVERGED: best value is still improving and full diversity remains -- continue running more generations.



**Interpretation (Q1):** the best fitness jumped from **729 → 961** and every chromosome in
Population 2 is still distinct (diversity = 4/4). Both signals say the search is still actively
improving, so with only two generations completed **Q1 has not yet converged** — more generations
are needed (in the earlier multi-generation run, this same setup converged to the true global
optimum $x=31,\ f(x)=961$ within about 3 generations).
